In [ ]:
from google.colab import files

# Upload your input file here.
# Expected: disease_clusters.xlsx
# It must have a column containing normalized disease names.
uploaded = files.upload()


Saving disease_clusters.xlsx to disease_clusters.xlsx


In [ ]:
!pip install pandas openpyxl tqdm google-cloud-aiplatform -q


In [ ]:
import pandas as pd
import json
import time
import os
from tqdm import tqdm

# ── STEP 1: Authenticate ────────────────────────────────────────────────────
# Each collaborator logs in with their own Gmail.
# Costs are charged to the project owner's Vertex AI credits.
from google.colab import auth
auth.authenticate_user()
print('Authenticated!')

# ── STEP 2: Init Vertex AI ──────────────────────────────────────────────────
import vertexai
from vertexai.generative_models import GenerativeModel

PROJECT_ID = 'project-7f940e6f-ba1d-404b-948'
LOCATION   = 'global'
MODEL_ID   = 'gemini-2.5-flash'

vertexai.init(project=PROJECT_ID, location=LOCATION)
model = GenerativeModel(MODEL_ID)
print(f'Vertex AI ready  |  project: {PROJECT_ID}  |  model: {MODEL_ID}')

# ── STEP 3: Sanity test ─────────────────────────────────────────────────────
test_response = model.generate_content('Reply with exactly the word: OK')
assert 'OK' in test_response.text.strip().upper(), f'Unexpected response: {test_response.text}'
print(f'Sanity check passed: {test_response.text.strip()}')

# ── CONFIG ──────────────────────────────────────────────────────────────────
INPUT_FILE       = 'disease_clusters.xlsx'
OUTPUT_FILE      = 'clustered_diseases_final.xlsx'
CHECKPOINT_FILE  = 'clustering_checkpoint.json'  # saves progress batch by batch
BATCH_SIZE       = 25   # 25 diseases per API call
MAX_RETRIES      = 3    # retries per batch on JSON parse failure
SLEEP_BETWEEN    = 1.0  # seconds between batches


Authenticated!
Vertex AI ready  |  project: project-7f940e6f-ba1d-404b-948  |  model: gemini-2.5-flash
Sanity check passed: OK


In [ ]:
# ── Load input file ─────────────────────────────────────────────────────────
df_input = pd.read_excel(INPUT_FILE)
print(f'Input file rows  : {len(df_input)}')
print(f'Input columns    : {list(df_input.columns)}')

# ── Auto-detect disease column ───────────────────────────────────────────────
DISEASE_COL = None
for candidate in ['normalized_disease', 'disease', 'Disease', 'Normalized_Disease', 'name']:
    if candidate in df_input.columns:
        DISEASE_COL = candidate
        break
if DISEASE_COL is None:
    DISEASE_COL = df_input.columns[0]
    print(f'Warning: no standard column found — using first column: {DISEASE_COL}')
else:
    print(f'Disease column   : {DISEASE_COL}')

# ── Extract UNIQUE normalized diseases ──────────────────────────────────────
# We cluster UNIQUE concepts only, then merge back to full dataset.
# This avoids sending duplicate concepts to the LLM.
unique_diseases = (
    df_input[DISEASE_COL]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
)
unique_diseases = sorted(unique_diseases)

# Remove empty strings
unique_diseases = [d for d in unique_diseases if d and d != 'nan']

print(f'Total input rows : {len(df_input)}')
print(f'Unique diseases  : {len(unique_diseases)}')
print(f'\nSample (first 15):')
for d in unique_diseases[:15]:
    print(f'  {d}')


Input file rows  : 3577
Input columns    : ['raw_disease_ar', 'disease_en', 'normalized_disease', 'frequency', 'cluster_id', 'broader_category', 'cluster_label', 'notes']
Disease column   : normalized_disease
Total input rows : 3577
Unique diseases  : 3577

Sample (first 15):
  abdominal and intestinal pain (colic)
  abdominal borborygmi, dyspepsia, and flatulence
  abdominal colic
  abdominal cramps
  abdominal distension
  abdominal flatulence
  abdominal flatulence, pruritus, scabies, ulcers, skin lesions, mania, leprosy, obsessive-compulsive disorder, hemorrhoids, and fissures
  abdominal pain
  abdominal pain (colic)
  abnormal bodily moistures
  abnormal heat (extraneous heat)
  abnormal humors (abnormal body fluids)
  abortion induction
  abrasion (skin excoriation)
  abrasions


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CLUSTERING FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════
#
# Key design decisions:
#   1. LLM returns JSON array, never CSV — zero comma-corruption risk
#   2. json.loads() is the only parser used — no string splitting
#   3. Schema validation on every record before accepting it
#   4. Automatic retry (up to MAX_RETRIES) on malformed JSON
#   5. Markdown fence stripping before parsing
#   6. Broad cluster vocabulary enforced in the prompt
#   7. Cluster label canonicalization after all batches complete

# ── Predefined broad cluster vocabulary ─────────────────────────────────────
# The LLM is instructed to pick from this list first.
# Only create a new label if truly none of these fit.
BROAD_CLUSTERS = [
    'respiratory_conditions',
    'gastrointestinal_conditions',
    'dermatological_conditions',
    'neurological_conditions',
    'musculoskeletal_conditions',
    'cardiovascular_conditions',
    'reproductive_conditions',
    'urological_conditions',
    'inflammatory_conditions',
    'infectious_conditions',
    'metabolic_conditions',
    'psychological_conditions',
    'ophthalmological_conditions',
    'otolaryngological_conditions',
    'wound_healing',
    'pain_management',
    'fever_and_systemic_illness',
    'nutritional_deficiency',
    'hepatic_conditions',
    'renal_conditions',
    'oral_dental_conditions',
    'hair_scalp_conditions',
    'endocrine_conditions',
    'haematological_conditions',
    'poisoning_and_toxicity',
    'general_weakness_fatigue',
    'parasitic_conditions',
    'oncological_conditions',
    'oedema_and_fluid_conditions',
    'humoral_conditions',          # for historical Arabic humoral medicine terms
]

BROAD_CATEGORY_MAP = {
    'respiratory_conditions':       'respiratory',
    'gastrointestinal_conditions':  'gastrointestinal',
    'dermatological_conditions':    'dermatological',
    'neurological_conditions':      'neurological',
    'musculoskeletal_conditions':   'musculoskeletal',
    'cardiovascular_conditions':    'cardiovascular',
    'reproductive_conditions':      'reproductive',
    'urological_conditions':        'urological',
    'inflammatory_conditions':      'inflammatory',
    'infectious_conditions':        'infectious',
    'metabolic_conditions':         'metabolic',
    'psychological_conditions':     'psychological',
    'ophthalmological_conditions':  'ophthalmological',
    'otolaryngological_conditions': 'otolaryngological',
    'wound_healing':                'wound',
    'pain_management':              'pain',
    'fever_and_systemic_illness':   'fever',
    'nutritional_deficiency':       'nutritional',
    'hepatic_conditions':           'hepatic',
    'renal_conditions':             'renal',
    'oral_dental_conditions':       'oral',
    'hair_scalp_conditions':        'dermatological',
    'endocrine_conditions':         'endocrine',
    'haematological_conditions':    'haematological',
    'poisoning_and_toxicity':       'toxic',
    'general_weakness_fatigue':     'general',
    'parasitic_conditions':         'parasitic',
    'oncological_conditions':       'oncological',
    'oedema_and_fluid_conditions':  'haematological',
    'humoral_conditions':           'general',
}

CLUSTER_LIST_STR = '\n'.join(f'  - {c}' for c in BROAD_CLUSTERS)

REQUIRED_KEYS = {'disease', 'cluster_label', 'broader_category'}


def strip_markdown_fences(text: str) -> str:
    """Remove ```json ... ``` or ``` ... ``` wrappers the LLM sometimes adds."""
    text = text.strip()
    if text.startswith('```'):
        # Remove first line (```json or ```)
        text = text[text.index('\n') + 1:] if '\n' in text else text[3:]
    if text.endswith('```'):
        text = text[:text.rfind('```')]
    return text.strip()


def validate_record(record: dict) -> bool:
    """Check a single JSON record has the required keys and non-empty string values."""
    if not isinstance(record, dict):
        return False
    if not REQUIRED_KEYS.issubset(record.keys()):
        return False
    for key in REQUIRED_KEYS:
        if not isinstance(record[key], str) or not record[key].strip():
            return False
    return True


def call_llm(batch: list) -> str | None:
    """Send a batch of diseases to the LLM. Returns raw text or None on API error."""
    disease_list = json.dumps(batch, ensure_ascii=False)

    prompt = (
        'You are building a biomedical SEARCH ONTOLOGY for literature retrieval.\n'
        'These disease terms come from historical Arabic herbal medicine texts.\n'
        'Some may use classical humoral terminology (black bile, phlegm, etc.).\n\n'

        'YOUR ONLY JOB: assign each disease a BROAD REUSABLE SEARCH GROUP\n'
        'so that many diseases can share one PubMed literature search.\n\n'

        'CRITICAL RULES:\n'
        '1. Use the BROADEST cluster that is medically sensible.\n'
        '2. The ENTIRE dataset will have only 30-60 unique cluster labels total.\n'
        '   This means MANY diseases in this batch MUST get the SAME cluster label.\n'
        '3. ALWAYS prefer a label from this predefined list:\n'
        f'{CLUSTER_LIST_STR}\n'
        '4. Only invent a NEW label if truly none of the above fit AT ALL.\n'
        '   New labels must still be broad snake_case, e.g. splenic_conditions.\n'
        '5. broader_category must be ONE word from this exact set:\n'
        '   respiratory, gastrointestinal, dermatological, neurological,\n'
        '   musculoskeletal, cardiovascular, reproductive, urological,\n'
        '   inflammatory, infectious, metabolic, psychological,\n'
        '   ophthalmological, otolaryngological, wound, pain, fever,\n'
        '   hepatic, renal, oral, endocrine, haematological, nutritional,\n'
        '   parasitic, toxic, oncological, general\n'
        '6. For ambiguous or humoral/historical terms, use the BROADER category.\n\n'

        'EXAMPLES OF CORRECT BROAD GROUPING:\n'
        '  cough, dyspnea, asthma, chest tightness → respiratory_conditions / respiratory\n'
        '  nausea, vomiting, diarrhea, bloating → gastrointestinal_conditions / gastrointestinal\n'
        '  black bile excess, phlegm disorder, humoral imbalance → humoral_conditions / general\n\n'

        'EXAMPLES OF WRONG OVER-SPECIFIC LABELS (NEVER DO THIS):\n'
        '  cough_related_conditions, chronic_productive_cough_disorder,\n'
        '  dyspnea_specific_cluster, upper_respiratory_inflammatory_distress\n\n'

        'OUTPUT FORMAT:\n'
        'Return ONLY a valid JSON array. No markdown. No explanation. No extra text.\n'
        'Each element must have exactly these three keys:\n'
        '  "disease"          — the disease string exactly as given in input\n'
        '  "cluster_label"    — broad snake_case cluster from the list above\n'
        '  "broader_category" — single word from the category set above\n\n'

        'Example output format:\n'
        '[\n'
        '  {"disease": "cough", "cluster_label": "respiratory_conditions", "broader_category": "respiratory"},\n'
        '  {"disease": "black bile excess", "cluster_label": "humoral_conditions", "broader_category": "general"}\n'
        ']\n\n'

        f'Diseases to cluster:\n{disease_list}\n'
    )

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        err = str(e)
        if '404' in err:
            print('\nERROR 404 — Vertex AI API not enabled or wrong model ID.')
            print('  Enable: https://console.cloud.google.com/apis/library/aiplatform.googleapis.com')
        elif '403' in err or 'PERMISSION_DENIED' in err:
            print('\nERROR 403 — Your account lacks Editor access on this project.')
            print('  Ask the owner to add your Gmail in IAM.')
        elif 'quota' in err.lower() or '429' in err:
            print('\nERROR 429 — Rate limit hit. Sleeping 30s before retry...')
            time.sleep(30)
        else:
            print(f'\nAPI ERROR: {e}')
        return None


def parse_json_response(raw_text: str, batch: list) -> list:
    """
    Parse the LLM JSON response.
    - Strips markdown fences
    - Uses json.loads() only — NEVER string splitting
    - Validates each record's schema
    - Normalises cluster_label and broader_category to lowercase stripped
    - Returns list of valid dicts; skips invalid records with a warning
    """
    if not raw_text:
        return []

    cleaned = strip_markdown_fences(raw_text)

    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f'\n  JSON parse error: {e}')
        print(f'  Raw text snippet: {raw_text[:300]}')
        return []  # caller will retry

    if not isinstance(data, list):
        print(f'  Expected JSON array, got {type(data)}')
        return []

    valid_rows = []
    for record in data:
        if not validate_record(record):
            print(f'  Skipping invalid record: {record}')
            continue
        valid_rows.append({
            'normalized_disease': record['disease'].strip().lower(),
            'cluster_label':      record['cluster_label'].strip().lower().replace(' ', '_'),
            'broader_category':   record['broader_category'].strip().lower().replace(' ', '_'),
        })

    return valid_rows


def cluster_batch_with_retry(batch: list) -> list:
    """
    Call the LLM with retry logic.
    Retries up to MAX_RETRIES times on JSON parse failure.
    Returns list of valid row dicts (may be partial on final failure).
    """
    for attempt in range(1, MAX_RETRIES + 1):
        raw = call_llm(batch)
        if raw is None:
            # API-level error — no point retrying immediately
            return []
        rows = parse_json_response(raw, batch)
        if rows:
            return rows
        if attempt < MAX_RETRIES:
            print(f'  Retry {attempt}/{MAX_RETRIES} for this batch...')
            time.sleep(2)
    print(f'  Batch failed after {MAX_RETRIES} retries. Diseases will be missing from output.')
    return []


print('All clustering functions defined.')
print(f'Broad cluster vocabulary: {len(BROAD_CLUSTERS)} predefined labels')


All clustering functions defined.
Broad cluster vocabulary: 30 predefined labels


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PROCESS BATCHES WITH CHECKPOINTING
# ════════════════════════════════════════════════════════════════════════════
#
# Checkpointing means: if the runtime crashes mid-run, re-running this cell
# picks up from where it left off instead of restarting from scratch.
# Progress is saved to CHECKPOINT_FILE after every batch.

# ── Load existing checkpoint if present ─────────────────────────────────────
checkpoint = {}
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)
    print(f'Checkpoint loaded: {len(checkpoint)} diseases already clustered.')
else:
    print('No checkpoint found — starting fresh.')

# ── Build list of diseases still needing clustering ──────────────────────────
remaining = [d for d in unique_diseases if d not in checkpoint]
print(f'Diseases to cluster this run : {len(remaining)}')
print(f'Already done (from checkpoint): {len(checkpoint)}')
print(f'Total unique diseases        : {len(unique_diseases)}')

failed_batches = []

# ── Process in batches ───────────────────────────────────────────────────────
batches = [remaining[i:i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
print(f'Batches to process: {len(batches)} x ~{BATCH_SIZE} diseases each\n')

for batch_idx, batch in enumerate(tqdm(batches, desc='Clustering')):
    rows = cluster_batch_with_retry(batch)

    if rows:
        for row in rows:
            checkpoint[row['normalized_disease']] = {
                'cluster_label':    row['cluster_label'],
                'broader_category': row['broader_category'],
            }
        # Save checkpoint after every batch
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)
    else:
        failed_batches.append(batch)

    time.sleep(SLEEP_BETWEEN)

print(f'\n────────────────────────────────')
print(f'Clustering complete.')
print(f'Total clustered  : {len(checkpoint)} / {len(unique_diseases)}')
print(f'Failed batches   : {len(failed_batches)}')
if failed_batches:
    flat_failed = [d for b in failed_batches for d in b]
    print(f'Failed diseases ({len(flat_failed)} total) — re-run this cell to retry them.')
    for d in flat_failed[:20]:
        print(f'  {d}')


No checkpoint found — starting fresh.
Diseases to cluster this run : 3577
Already done (from checkpoint): 0
Total unique diseases        : 3577
Batches to process: 144 x ~25 diseases each



Clustering: 100%|██████████| 144/144 [58:08<00:00, 24.23s/it]


────────────────────────────────
Clustering complete.
Total clustered  : 3577 / 3577
Failed batches   : 0


In [ ]:
import difflib

def canonicalize_cluster(label: str) -> str:
    """Map a cluster label to the closest predefined broad cluster."""
    label = label.strip().lower().replace(' ', '_')

    # Exact match first
    if label in BROAD_CLUSTERS:
        return label

    # Strip common suffix variants and try again
    for suffix in ['_disorders', '_disorder', '_disease', '_diseases',
                   '_condition', '_issues', '_problem', '_problems',
                   '_symptoms', '_related', '_cluster']:
        if label.endswith(suffix):
            stem = label[:-len(suffix)] + '_conditions'
            if stem in BROAD_CLUSTERS:
                return stem

    # Fuzzy match against predefined list
    matches = difflib.get_close_matches(label, BROAD_CLUSTERS, n=1, cutoff=0.70)
    if matches:
        return matches[0]

    # Substring containment fallback
    for broad in BROAD_CLUSTERS:
        core = broad.replace('_conditions', '').replace('_management', '')
        if core in label or label in core:
            return broad

    # Cannot map — keep the label as-is
    return label


# Apply canonicalization to every disease in checkpoint
canon_map = {}   # disease → {cluster_label, broader_category}
drift_fixed = 0

for disease, info in checkpoint.items():
    original_label = info['cluster_label']
    canonical      = canonicalize_cluster(original_label)

    # Derive broader_category from canonical label using the map,
    # falling back to whatever the LLM originally returned.
    broad_cat = BROAD_CATEGORY_MAP.get(canonical, info['broader_category'])

    if canonical != original_label:
        drift_fixed += 1

    canon_map[disease] = {
        'cluster_label':    canonical,
        'broader_category': broad_cat,
    }

print(f'Cluster labels canonicalized  : {len(canon_map)}')
print(f'Drift corrections applied     : {drift_fixed}')
print(f'Unique cluster labels (after) : {len(set(v["cluster_label"] for v in canon_map.values()))}')
print(f'Unique broader categories     : {len(set(v["broader_category"] for v in canon_map.values()))}')

# ════════════════════════════════════════════════════════════════════════════
# STEP 2: BUILD CLUSTERED UNIQUE-DISEASE TABLE
# ════════════════════════════════════════════════════════════════════════════

cluster_df = pd.DataFrame([
    {
        'normalized_disease': disease,
        'cluster_label':      info['cluster_label'],
        'broader_category':   info['broader_category'],
    }
    for disease, info in canon_map.items()
])
cluster_df = cluster_df.sort_values(['broader_category', 'cluster_label', 'normalized_disease'])
cluster_df = cluster_df.reset_index(drop=True)

print(f'\nCluster distribution (top 20 by disease count):')
print(cluster_df['cluster_label'].value_counts().head(20).to_string())

# ════════════════════════════════════════════════════════════════════════════
# STEP 3: MERGE CLUSTER LABELS BACK TO FULL INPUT DATASET
# ════════════════════════════════════════════════════════════════════════════
#
# The input file may have many rows per disease (multiple herbs/claims).
# We join cluster labels back so every original row gets its cluster.

df_merge = df_input.copy()
df_merge['_disease_key'] = df_merge[DISEASE_COL].astype(str).str.strip().str.lower()

# Drop existing 'cluster_label' and 'broader_category' from df_merge
# to avoid column overlap during join, as we are replacing them with
# the canonicalized values from 'lookup'.
if 'cluster_label' in df_merge.columns:
    df_merge = df_merge.drop(columns=['cluster_label'])
if 'broader_category' in df_merge.columns:
    df_merge = df_merge.drop(columns=['broader_category'])

lookup = cluster_df.set_index('normalized_disease')[['cluster_label', 'broader_category']]

df_merge = df_merge.join(lookup, on='_disease_key', how='left')
df_merge = df_merge.drop(columns=['_disease_key'])

unmatched = df_merge['cluster_label'].isna().sum()
print(f'\nFull dataset rows             : {len(df_merge)}')
print(f'Rows with cluster assigned    : {df_merge["cluster_label"].notna().sum()}')
print(f'Rows without cluster (failed) : {unmatched}')

# ════════════════════════════════════════════════════════════════════════════
# STEP 4: SAVE
# ════════════════════════════════════════════════════════════════════════════

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df_merge.to_excel(writer, sheet_name='Full_Dataset', index=False)
    cluster_df.to_excel(writer, sheet_name='Cluster_Reference', index=False)

print(f'\nSaved: {OUTPUT_FILE}')
print(f'  Sheet 1 — Full_Dataset      : {len(df_merge)} rows (all original rows + cluster columns)')
print(f'  Sheet 2 — Cluster_Reference : {len(cluster_df)} rows (unique diseases + clusters)')
print(f'\nFinal stats:')
print(f'  Unique cluster labels       : {cluster_df["cluster_label"].nunique()}')
print(f'  Unique broader categories   : {cluster_df["broader_category"].nunique()}')
print(f'\nBroader category breakdown:')
print(cluster_df.groupby("broader_category")["cluster_label"].nunique().sort_values(ascending=False).to_string())

Cluster labels canonicalized  : 3577
Drift corrections applied     : 5
Unique cluster labels (after) : 31
Unique broader categories     : 27

Cluster distribution (top 20 by disease count):
cluster_label
humoral_conditions              566
gastrointestinal_conditions     310
dermatological_conditions       307
reproductive_conditions         246
respiratory_conditions          195
neurological_conditions         181
ophthalmological_conditions     148
psychological_conditions        136
poisoning_and_toxicity          129
general_weakness_fatigue        114
wound_healing                   105
fever_and_systemic_illness      104
musculoskeletal_conditions      103
inflammatory_conditions          91
urological_conditions            90
otolaryngological_conditions     89
oral_dental_conditions           89
hair_scalp_conditions            78
oncological_conditions           68
pain_management                  61

Full dataset rows             : 3577
Rows with cluster assigned    : 3577
R

In [ ]:
from google.colab import files

files.download(OUTPUT_FILE)
print(f'Downloaded: {OUTPUT_FILE}')

# Optionally also download the checkpoint for safekeeping
# files.download(CHECKPOINT_FILE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: clustered_diseases_final.xlsx
